In [1]:
#Reload your cleaned dataset (fresh start for the modeling
import pandas as pd

df = pd.read_csv('data/processed/db_delays_cleaned.csv', parse_dates=[
    'arrival_plan', 'departure_plan', 'arrival_change', 'departure_change'
])
df.shape

(2061357, 26)

In [2]:
#Build dim_station
dim_station = (
    df[['eva_nr', 'station', 'state', 'city', 'zip', 'lat', 'long', 'category']]
    .drop_duplicates(subset='eva_nr')
    .reset_index(drop=True)
)
dim_station['station_id'] = dim_station.index + 1

print(dim_station.shape)
dim_station.head()

(1996, 9)


,eva_nr,station,state,city,zip,lat,long,category,station_id
0,8000001,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,50.767800,6.091499,2,1
1,8000406,Aachen-Rothe Erde,Nordrhein-Westfalen,Aachen,52066,50.770202,6.116475,4,2
2,8000404,Aachen West,Nordrhein-Westfalen,Aachen,52072,50.780360,6.070715,5,3
3,8000002,Aalen Hbf,Baden-Württemberg,Aalen,73430,48.841013,10.096271,3,4
4,8000413,Achim,Niedersachsen,Achim,28832,53.015990,9.030447,4,5


In [3]:
# Building dim_line
df['train_type'] = df['line'].str.extract(r'^([A-Za-z]+)')

dim_line = (
    df[['line', 'train_type']]
    .drop_duplicates(subset='line')
    .reset_index(drop=True)
)
dim_line['line_id'] = dim_line.index + 1

print(dim_line.shape)
dim_line.head()

(296, 3)


,line,train_type,line_id
0,20,NaN,1
1,18,NaN,2
2,1,NaN,3
3,33,NaN,4
4,4,NaN,5


In [4]:
#building dim_date 
df['arrival_plan'] = pd.to_datetime(df['arrival_plan'], errors='coerce')

In [5]:
dim_date = pd.DataFrame({'date': df['arrival_plan'].dt.date.unique()}).sort_values('date').reset_index(drop=True)
dim_date['date_id'] = dim_date.index + 1
dim_date['weekday_name'] = pd.to_datetime(dim_date['date']).dt.day_name()
dim_date['is_weekend'] = pd.to_datetime(dim_date['date']).dt.weekday >= 5
dim_date

,date,date_id,weekday_name,is_weekend
0,2024-07-07,1,Sunday,True
1,2024-07-08,2,Monday,False
2,2024-07-09,3,Tuesday,False
3,2024-07-10,4,Wednesday,False
4,2024-07-11,5,Thursday,False
5,2024-07-12,6,Friday,False
6,2024-07-13,7,Saturday,True
7,2024-07-14,8,Sunday,True
8,NaT,9,NaN,False


In [6]:
# confirm all show datetime64
time_cols = ['arrival_plan', 'departure_plan', 'arrival_change', 'departure_change']
for col in time_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df[time_cols].dtypes  

arrival_plan        datetime64[ns]
departure_plan      datetime64[ns]
arrival_change      datetime64[ns]
departure_change    datetime64[ns]
dtype: object

In [7]:
#Build fact_delays — merge FKs back in, keep only measures + keys
fact_delays = df.merge(dim_station[['eva_nr', 'station_id']], on='eva_nr', how='left')
fact_delays = fact_delays.merge(dim_line[['line', 'line_id']], on='line', how='left')

fact_delays['date'] = fact_delays['arrival_plan'].dt.date
fact_delays = fact_delays.merge(dim_date[['date', 'date_id']], on='date', how='left')

fact_delays['hour'] = fact_delays['arrival_plan'].dt.hour

fact_delays = fact_delays[[
    'ID', 'station_id', 'line_id', 'date_id', 'hour',
    'arrival_plan', 'arrival_change', 'departure_plan', 'departure_change',
    'arrival_delay_m', 'departure_delay_m',
    'arrival_delay_check', 'departure_delay_check', 'info'
]].rename(columns={'ID': 'stop_id'})

print(fact_delays.shape)
fact_delays.head()

(2061357, 14)


,stop_id,station_id,line_id,date_id,hour,arrival_plan,arrival_change,departure_plan,departure_change,arrival_delay_m,departure_delay_m,arrival_delay_check,departure_delay_check,info
0,1573967790757085557-2407072312-14,1,1,2,0.0,2024-07-08 00:00:00,2024-07-08 00:03:00,2024-07-08 00:01:00,2024-07-08 00:04:00,3,3,on_time,on_time,0
1,349781417030375472-2407080017-1,1,2,9,NaN,NaT,NaT,2024-07-08 00:17:00,NaT,0,0,on_time,on_time,0
2,7157250219775883918-2407072120-25,2,3,2,0.0,2024-07-08 00:03:00,2024-07-08 00:03:00,2024-07-08 00:04:00,2024-07-08 00:04:00,0,0,on_time,on_time,0
3,349781417030375472-2407080017-2,3,2,2,0.0,2024-07-08 00:20:00,NaT,2024-07-08 00:21:00,NaT,0,0,on_time,on_time,0
4,1983158592123451570-2407080010-3,3,4,2,0.0,2024-07-08 00:20:00,2024-07-08 00:20:00,2024-07-08 00:21:00,2024-07-08 00:21:00,0,0,on_time,on_time,0


In [8]:
#Sanity check no broken foreign key
print(fact_delays['station_id'].isna().sum()) 
print(fact_delays['line_id'].isna().sum())    
print(fact_delays['date_id'].isna().sum())     

0
0
0


In [10]:
#saving all four tables
dim_station.to_csv('data/processed/dim_station.csv', index=False)
dim_line.to_csv('data/processed/dim_line.csv', index=False)
dim_date.to_csv('data/processed/dim_date.csv', index=False)
fact_delays.to_csv('data/processed/fact_delays.csv', index=False)